# GOYU General - KoAlpaca Training

This notebook trains the GOYU General model using a sliced version of the `beomi/KoAlpaca-v1.1a` dataset.

In [ ]:
!pip install datasets sentencepiece torch

In [ ]:
!git clone https://github.com/dodo701/goyu-ai.git
%cd goyu-ai/goyu-general

## 1. Load Dataset and Prepare Tokenizer

In [ ]:
from datasets import load_dataset

# Load KoAlpaca dataset
dataset = load_dataset('beomi/KoAlpaca-v1.1a', split='train')

# Slice the first 400 examples
dataset = dataset.select(range(400))

# Format data into a text file for tokenizer training
temp_text_file = 'temp_corpus.txt'
with open(temp_text_file, 'w', encoding='utf-8') as f:
    for row in dataset:
        # We use the prompt format: User: {instruction}\nAssistant: {output}
        f.write(f"User: {row['instruction']}\n")
        f.write(f"Assistant: {row['output']}\n")


In [ ]:
from src.tokenizer import train_tokenizer, GoyuTokenizer
import os

# Train tokenizer
os.makedirs('data/processed', exist_ok=True)
tokenizer_prefix = 'data/processed/tokenizer'
train_tokenizer(temp_text_file, tokenizer_prefix, vocab_size=8000) # smaller vocab for 400 items

tokenizer = GoyuTokenizer(f'{tokenizer_prefix}.model')
print(f"Tokenizer trained. Vocab size: {tokenizer.vocab_size}")


## 2. Prepare Data for Model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import json

class KoAlpacaDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=512):
        self.data = []
        for row in hf_dataset:
            instruction = row['instruction']
            output = row['output']
            
            # Prompt format
            prompt = f"User: {instruction}\nAssistant: "
            full_text = prompt + output
            
            # Encode
            full_ids = tokenizer.encode(full_text, add_bos=True, add_eos=True)
            prompt_ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)
            
            if len(full_ids) > max_length:
                full_ids = full_ids[:max_length]
                
            # Create labels (ignore prompt for loss calculation by setting to -100)
            labels = full_ids.copy()
            labels[:len(prompt_ids)] = [-100] * min(len(prompt_ids), len(labels))
            
            self.data.append({
                'input_ids': torch.tensor(full_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long)
            })
            
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    labels = [item['labels'] for item in batch]
    
    # Pad sequences
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_id)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return input_ids, labels

train_dataset = KoAlpacaDataset(dataset, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)


## 3. Train Model

In [ ]:
from src.model import GoyuGeneral, ModelConfig
import torch.optim as optim
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('config/base.json', 'r') as f:
    config_dict = json.load(f)
    
config_dict['vocab_size'] = tokenizer.vocab_size
config = ModelConfig(**config_dict)

model = GoyuGeneral(config).to(device)

optimizer = optim.AdamW(model.parameters(), lr=config_dict.get('learning_rate', 3e-4))

epochs = 3
model.train()

for epoch in range(epochs):
    total_loss = 0
    start_time = time.time()
    for batch_idx, (input_ids, labels) in enumerate(train_loader):
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits, loss = model(input_ids, labels=labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f'Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
            
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1} completed in {time.time()-start_time:.2f}s | Average Loss: {avg_loss:.4f}\n')


## 4. Save Model

In [ ]:
import os

os.makedirs('models', exist_ok=True)
save_path = 'models/koalpaca_model.pt'

torch.save({
    'model': model.state_dict(),
    'config': config_dict
}, save_path)

print(f"Model saved to {save_path}")


## 5. Inference and Download

In [ ]:
import torch.nn.functional as F

model.eval()
test_instruction = "슈퍼맨에 대해서 설명해줘."
prompt = f"User: {test_instruction}\nAssistant: "

input_ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)
x = torch.tensor([input_ids], dtype=torch.long, device=device)

max_new_tokens = 50
temperature = 0.7

with torch.no_grad():
    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :]
        
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        
        x = torch.cat((x, next_id), dim=1)
        if next_id.item() == tokenizer.eos_id:
            break
            
generated_ids = x[0].tolist()[len(input_ids):]
response = tokenizer.decode(generated_ids)
print("Prompt:", prompt)
print("Response:", response)


In [ ]:
from google.colab import files
files.download(save_path)
# To download the tokenizer as well:
files.download('data/processed/tokenizer.model')
